# NEXUS LLM — Module 4: Next Token Prediction Lab

Complete notebook for inspecting DistilGPT2 next-token predictions.

Topics: tokenization, logits, Softmax, top predictions, greedy decoding, temperature, Top-K, Top-P, sampling, visualization and one autoregressive step.

Run cells from top to bottom.

## 1. Imports and configuration

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilbert/distilgpt2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 2. Load tokenizer and causal language model

`AutoModelForCausalLM` includes the language-modeling head needed to produce vocabulary logits.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model loaded")
print("Vocabulary size:", model.config.vocab_size)
print("Hidden size:", model.config.n_embd)
print("Transformer layers:", model.config.n_layer)
print("Attention heads:", model.config.n_head)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded
Vocabulary size: 50257
Hidden size: 768
Transformer layers: 6
Attention heads: 12


## 3. Tokenize input text

The model receives token IDs, not raw text.

In [3]:
text = "Artificial intelligence is"
inputs = tokenizer(text, return_tensors="pt")
inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
input_ids = inputs["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

print("Input:", text)
print("Tokens:", tokens)
print("Token IDs:", input_ids)
print("Input shape:", tuple(input_ids.shape))

Input: Artificial intelligence is
Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis']
Token IDs: tensor([[8001, 9542, 4430,  318]])
Input shape: (1, 4)


## 4. Extract logits

Logits have shape `(batch, sequence_length, vocabulary_size)`. The final input position produces the scores for the next token.

In [4]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
next_token_logits = logits[:, -1, :]
print("Logits shape:", tuple(logits.shape))
print("Next-token logits shape:", tuple(next_token_logits.shape))

Logits shape: (1, 4, 50257)
Next-token logits shape: (1, 50257)


## 5. Softmax probabilities

Softmax converts raw logits into a probability distribution across the vocabulary.

In [5]:
probabilities = F.softmax(next_token_logits, dim=-1)
print("Probability shape:", tuple(probabilities.shape))
print("Probability sum:", probabilities.sum(dim=-1).item())

Probability shape: (1, 50257)
Probability sum: 1.000011682510376


## 6. Top predicted tokens

In [6]:
def get_top_predictions(probability_tensor, tokenizer, top_n=10):
    top_n = min(top_n, probability_tensor.shape[-1])
    top_probs, top_ids = torch.topk(probability_tensor[0], k=top_n)
    rows = []
    for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
        rows.append({
            "token_id": token_id,
            "token": repr(tokenizer.decode([token_id])),
            "probability": probability,
            "probability_percent": probability * 100
        })
    return pd.DataFrame(rows)

prediction_df = get_top_predictions(probabilities, tokenizer, top_n=10)
display(prediction_df)

,token_id,token,probability,probability_percent
0,257,' a',0.164362,16.436212
1,262,' the',0.066132,6.613249
2,407,' not',0.048673,4.867272
3,530,' one',0.036778,3.677796
4,281,' an',0.035226,3.522589
5,783,' now',0.016249,1.624936
6,5033,' becoming',0.015808,1.580811
7,635,' also',0.014690,1.469033
8,655,' just',0.011326,1.132585
9,991,' still',0.011237,1.123651


## 7. Greedy decoding

Greedy decoding selects the highest-probability token.

In [7]:
greedy_token_id = torch.argmax(probabilities, dim=-1).item()
greedy_token = tokenizer.decode([greedy_token_id])
print("Greedy token ID:", greedy_token_id)
print("Greedy prediction:", repr(greedy_token))

Greedy token ID: 257
Greedy prediction: ' a'


## 8. Temperature

Temperature is applied to logits before Softmax. Lower values sharpen the distribution; higher values flatten it.

In [8]:
def apply_temperature(logits_tensor, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    return F.softmax(logits_tensor / temperature, dim=-1)

temperatures = [0.5, 1.0, 1.5]
temperature_probabilities = {}

for temperature in temperatures:
    temp_probs = apply_temperature(next_token_logits, temperature)
    temperature_probabilities[temperature] = temp_probs
    print(f"\\nTemperature = {temperature}")
    display(get_top_predictions(temp_probs, tokenizer, top_n=5))

\nTemperature = 0.5


,token_id,token,probability,probability_percent
0,257,' a',0.695876,69.587630
1,262,' the',0.112657,11.265706
2,407,' not',0.061024,6.102389
3,530,' one',0.034842,3.484206
4,281,' an',0.031963,3.196336


\nTemperature = 1.0


,token_id,token,probability,probability_percent
0,257,' a',0.164362,16.436212
1,262,' the',0.066132,6.613249
2,407,' not',0.048673,4.867272
3,530,' one',0.036778,3.677796
4,281,' an',0.035226,3.522589


\nTemperature = 1.5


,token_id,token,probability,probability_percent
0,257,' a',0.031936,3.193636
1,262,' the',0.017406,1.740581
2,407,' not',0.014189,1.418866
3,530,' one',0.011771,1.177089
4,281,' an',0.011437,1.143732


## 9. Top-K filtering

Top-K retains only the K highest logits; excluded candidates receive zero probability after Softmax.

In [9]:
def apply_top_k(logits_tensor, k=50):
    if k <= 0:
        raise ValueError("k must be greater than 0")
    k = min(k, logits_tensor.shape[-1])
    top_values, _ = torch.topk(logits_tensor, k=k, dim=-1)
    threshold = top_values[..., -1, None]
    return logits_tensor.masked_fill(logits_tensor < threshold, float("-inf"))

top_k_logits = apply_top_k(next_token_logits, k=50)
top_k_probabilities = F.softmax(top_k_logits, dim=-1)
print("Probability sum:", top_k_probabilities.sum().item())
print("Remaining candidates:", int((top_k_probabilities[0] > 0).sum().item()))
display(get_top_predictions(top_k_probabilities, tokenizer, top_n=10))

Probability sum: 0.9999998807907104
Remaining candidates: 50


,token_id,token,probability,probability_percent
0,257,' a',0.263838,26.383784
1,262,' the',0.106157,10.615739
2,407,' not',0.078131,7.813057
3,530,' one',0.059037,5.903682
4,281,' an',0.056545,5.654540
5,783,' now',0.026084,2.608385
6,5033,' becoming',0.025376,2.537554
7,635,' also',0.023581,2.358125
8,655,' just',0.018181,1.818052
9,991,' still',0.018037,1.803711


## 10. Top-P (nucleus) filtering

Top-P retains the smallest high-probability group whose cumulative probability reaches the threshold, including the token that crosses it.

In [10]:
def apply_top_p(logits_tensor, top_p=0.9):
    if not 0 < top_p <= 1:
        raise ValueError("top_p must be in the interval (0, 1]")

    sorted_logits, sorted_indices = torch.sort(logits_tensor, descending=True, dim=-1)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    sorted_remove = cumulative_probs > top_p
    sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
    sorted_remove[..., 0] = False

    remove_mask = torch.zeros_like(logits_tensor, dtype=torch.bool)
    remove_mask.scatter_(dim=-1, index=sorted_indices, src=sorted_remove)

    return logits_tensor.masked_fill(remove_mask, float("-inf"))

top_p_logits = apply_top_p(next_token_logits, top_p=0.90)
top_p_probabilities = F.softmax(top_p_logits, dim=-1)
print("Probability sum:", top_p_probabilities.sum().item())
print("Remaining candidates:", int((top_p_probabilities[0] > 0).sum().item()))
display(get_top_predictions(top_p_probabilities, tokenizer, top_n=10))

Probability sum: 1.0
Remaining candidates: 569


,token_id,token,probability,probability_percent
0,257,' a',0.182615,18.261482
1,262,' the',0.073477,7.347662
2,407,' not',0.054078,5.407791
3,530,' one',0.040862,4.086221
4,281,' an',0.039138,3.913778
5,783,' now',0.018054,1.805389
6,5033,' becoming',0.017564,1.756363
7,635,' also',0.016322,1.632171
8,655,' just',0.012584,1.258361
9,991,' still',0.012484,1.248435
